In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd


# 1. 读取 sta.csv（分钟级管网数据）

sta = pd.read_csv("sta.csv")

# 时间列转为 datetime
sta["Datetime"] = pd.to_datetime(sta["Datetime"])

# 设置时间索引并排序
sta = sta.set_index("Datetime").sort_index()

# 选取建模需要的列
sta = sta[
    [
        "WaterLevel(m)",      # 液位（预测目标）
        "Flow(L/s)",          # 流量
        "Velocity(m/s)",      # 流速
        "WaterTemperature(_)" # 水温
    ]
]

# 确保时间频率为 1 分钟
sta = sta.resample("1min").mean()



# 2. 读取 met.csv（小时级气象数据）

met = pd.read_csv("met.csv")

met["Datetime"] = pd.to_datetime(met["Datetime"])
met = met.set_index("Datetime").sort_index()

met = met[
    [
        "Precipiation(mm)",     # 降雨
        "AirTemperature(_)",    # 气温
        "RelativeHumidity(%)",  # 湿度
        "VaporPressure(kPa)",   # 水汽压
        "WindSpeed(m/s)",       # 风速
        "WindDirection"         # 风向
    ]
]

# 确保是小时级
met = met.resample("1h").mean()



# 3. 小时 → 分钟 对齐（关键步骤）

# 使用前向填充：一个小时内气象条件保持不变
met_aligned = met.reindex(sta.index, method="ffill")



# 4. 合并数据

df = pd.concat([sta, met_aligned], axis=1)



# 删除仍存在缺失值的时间点
df = df.dropna()

print(df.head())
print(df.shape)


/tmp/ipython-input-3618199970.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sta["Datetime"] = pd.to_datetime(sta["Datetime"])
/tmp/ipython-input-3618199970.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  met["Datetime"] = pd.to_datetime(met["Datetime"])


                     WaterLevel(m)  Flow(L/s)  Velocity(m/s)  \
Datetime                                                       
2022-09-19 16:00:00          2.287    238.258          0.474   
2022-09-19 16:01:00          2.283    233.734          0.465   
2022-09-19 16:02:00          2.287    233.734          0.465   
2022-09-19 16:03:00          2.283    238.258          0.474   
2022-09-19 16:04:00          2.283    229.211          0.456   

                     WaterTemperature(_)  Precipiation(mm)  AirTemperature(_)  \
Datetime                                                                        
2022-09-19 16:00:00               23.222               0.0               23.3   
2022-09-19 16:01:00               23.222               0.0               23.3   
2022-09-19 16:02:00               23.222               0.0               23.3   
2022-09-19 16:03:00               23.222               0.0               23.3   
2022-09-19 16:04:00               23.222               0.0       

In [ ]:
df.head(10)

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-09-19 16:00:00,2.287,238.258,0.474,23.222,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:01:00,2.283,233.734,0.465,23.222,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:02:00,2.287,233.734,0.465,23.222,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:03:00,2.283,238.258,0.474,23.222,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:04:00,2.283,229.211,0.456,23.222,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:05:00,2.283,233.734,0.465,23.295,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:06:00,2.283,224.687,0.447,23.295,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:07:00,2.283,220.163,0.438,23.295,0.0,23.3,36.0,1004.0,0.0,0.0
2022-09-19 16:08:00,2.283,229.211,0.456,23.295,0.0,23.3,36.0,1004.0,0.0,0.0


In [ ]:
df.tail(10)

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2023-03-16 17:22:00,0.336,74.529,0.372,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:23:00,0.332,80.848,0.410,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:24:00,0.332,80.848,0.410,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:25:00,0.332,84.595,0.429,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:26:00,0.332,86.369,0.438,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:27:00,0.332,82.820,0.420,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:28:00,0.332,88.144,0.447,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:29:00,0.327,82.906,0.429,20.473,0.0,5.7,50.0,1021.0,2.9,167.0
2023-03-16 17:30:00,0.332,80.848,0.410,20.473,0.0,5.7,50.0,1021.0,2.9,167.0


In [ ]:
df_train = df.loc["2022-10-01":"2022-11-11"]
df_forecast = df.loc["2022-11-11":"2022-12-01"]
df_reference = df.loc["2022-10-20":"2022-11-10"]
df_anomaly = df.loc["2023-01-01":"2023-02-01"]

In [ ]:
df_train.head()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-10-01 00:00:00,1.364,176.934,0.352,22.882,0.0,21.4,83.0,1007.0,0.0,0.0
2022-10-01 00:01:00,1.364,186.988,0.372,22.882,0.0,21.4,83.0,1007.0,0.0,0.0
2022-10-01 00:02:00,1.364,181.961,0.362,22.882,0.0,21.4,83.0,1007.0,0.0,0.0
2022-10-01 00:03:00,1.364,176.934,0.352,22.882,0.0,21.4,83.0,1007.0,0.0,0.0
2022-10-01 00:04:00,1.364,186.988,0.372,22.882,0.0,21.4,83.0,1007.0,0.0,0.0


In [ ]:
df_train.tail()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-11-11 23:55:00,0.823,275.957,0.549,19.155,0.0,17.4,95.0,1010.0,3.9,140.0
2022-11-11 23:56:00,0.818,279.979,0.557,19.155,0.0,17.4,95.0,1010.0,3.9,140.0
2022-11-11 23:57:00,0.823,279.979,0.557,19.155,0.0,17.4,95.0,1010.0,3.9,140.0
2022-11-11 23:58:00,0.818,275.957,0.549,19.155,0.0,17.4,95.0,1010.0,3.9,140.0
2022-11-11 23:59:00,0.814,279.979,0.557,19.155,0.0,17.4,95.0,1010.0,3.9,140.0


In [ ]:
df_forecast.head()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-11-11 00:00:00,0.859,284.000,0.565,19.011,0.0,17.1,93.0,1014.0,3.0,130.0
2022-11-11 00:01:00,0.855,284.000,0.565,19.011,0.0,17.1,93.0,1014.0,3.0,130.0
2022-11-11 00:02:00,0.855,279.979,0.557,19.011,0.0,17.1,93.0,1014.0,3.0,130.0
2022-11-11 00:03:00,0.850,275.957,0.549,19.011,0.0,17.1,93.0,1014.0,3.0,130.0
2022-11-11 00:04:00,0.850,267.915,0.533,19.011,0.0,17.1,93.0,1014.0,3.0,130.0


In [ ]:
df_forecast.tail()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-12-01 23:55:00,1.105,0.0,0.0,17.286,0.0,-1.4,32.0,1025.0,9.7,353.0
2022-12-01 23:56:00,1.105,0.0,0.0,17.286,0.0,-1.4,32.0,1025.0,9.7,353.0
2022-12-01 23:57:00,1.105,0.0,0.0,17.286,0.0,-1.4,32.0,1025.0,9.7,353.0
2022-12-01 23:58:00,1.100,0.0,0.0,17.286,0.0,-1.4,32.0,1025.0,9.7,353.0
2022-12-01 23:59:00,1.096,0.0,0.0,17.286,0.0,-1.4,32.0,1025.0,9.7,353.0


In [ ]:
df_reference.head()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-10-20 00:00:00,1.055,250.825,0.499,19.921,0.0,15.7,52.0,1018.0,0.0,0.0
2022-10-20 00:01:00,1.055,246.804,0.491,19.921,0.0,15.7,52.0,1018.0,0.0,0.0
2022-10-20 00:02:00,1.055,242.280,0.482,19.921,0.0,15.7,52.0,1018.0,0.0,0.0
2022-10-20 00:03:00,1.050,242.280,0.482,19.921,0.0,15.7,52.0,1018.0,0.0,0.0
2022-10-20 00:04:00,1.050,250.825,0.499,19.921,0.0,15.7,52.0,1018.0,0.0,0.0


In [ ]:
df_reference.tail()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2022-11-10 23:55:00,0.868,284.000,0.565,19.011,0.0,17.3,94.0,1014.0,2.9,127.0
2022-11-10 23:56:00,0.864,279.979,0.557,19.011,0.0,17.3,94.0,1014.0,2.9,127.0
2022-11-10 23:57:00,0.864,275.957,0.549,19.011,0.0,17.3,94.0,1014.0,2.9,127.0
2022-11-10 23:58:00,0.859,279.979,0.557,19.011,0.0,17.3,94.0,1014.0,2.9,127.0
2022-11-10 23:59:00,0.859,288.021,0.573,19.011,0.0,17.3,94.0,1014.0,2.9,127.0


In [ ]:
df_anomaly.head()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2023-01-01 00:00:00,0.555,20.840,0.056,26.764,0.0,2.7,72.0,1023.0,2.3,359.0
2023-01-01 00:01:00,0.550,0.000,0.000,26.764,0.0,2.7,72.0,1023.0,2.3,359.0
2023-01-01 00:02:00,0.550,11.790,0.032,26.764,0.0,2.7,72.0,1023.0,2.3,359.0
2023-01-01 00:03:00,0.550,36.108,0.098,26.764,0.0,2.7,72.0,1023.0,2.3,359.0
2023-01-01 00:04:00,0.550,67.427,0.183,26.764,0.0,2.7,72.0,1023.0,2.3,359.0


In [ ]:
df_anomaly.tail()

,WaterLevel(m),Flow(L/s),Velocity(m/s),WaterTemperature(_),Precipiation(mm),AirTemperature(_),RelativeHumidity(%),VaporPressure(kPa),WindSpeed(m/s),WindDirection
Datetime,,,,,,,,,,
2023-02-01 23:55:00,0.900,308.127,0.613,21.506,0.0,-1.2,43.0,1023.0,5.6,335.0
2023-02-01 23:56:00,0.900,304.106,0.605,21.506,0.0,-1.2,43.0,1023.0,5.6,335.0
2023-02-01 23:57:00,0.905,304.106,0.605,21.506,0.0,-1.2,43.0,1023.0,5.6,335.0
2023-02-01 23:58:00,0.900,296.064,0.589,21.506,0.0,-1.2,43.0,1023.0,5.6,335.0
2023-02-01 23:59:00,0.896,296.064,0.589,21.506,0.0,-1.2,43.0,1023.0,5.6,335.0


In [ ]:
df.to_csv("df.csv")
df_train.to_csv("df_train.csv")
df_forecast.to_csv("df_forecast.csv")
df_reference.to_csv("df_reference.csv")
df_anomaly.to_csv("df_anomaly.csv")


In [ ]:
files.download("df.csv")
files.download("df_train.csv")
files.download("df_forecast.csv")
files.download("df_reference.csv")
files.download("df_anomaly.csv")